In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np

from torchvision.models import resnet18, ResNet18_Weights
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


device = "cuda" if torch.cuda.is_available() else "cpu"


transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor()
])

train_data = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_data = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_data, batch_size=256, shuffle=False
)
test_loader = torch.utils.data.DataLoader(
    test_data, batch_size=256, shuffle=False
)


resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
resnet.fc = nn.Identity()
resnet.to(device)
resnet.eval()


def extract_embeddings(loader):
    X, y = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            feats = resnet(imgs)
            X.append(feats.cpu().numpy())
            y.append(labels.numpy())
    return np.vstack(X), np.hstack(y)

X_train, y_train = extract_embeddings(train_loader)
X_test, y_test = extract_embeddings(test_loader)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

lr = LogisticRegression(max_iter=5000)
svm = SVC(probability=True)
mlp = MLPClassifier(hidden_layer_sizes=(256,), max_iter=300)

lr.fit(X_train, y_train)
svm.fit(X_train, y_train)
mlp.fit(X_train, y_train)


p_lr = lr.predict_proba(X_test)
p_svm = svm.predict_proba(X_test)
p_mlp = mlp.predict_proba(X_test)


p_ensemble = (p_lr + p_svm + p_mlp) / 3
y_ensemble = np.argmax(p_ensemble, axis=1)


print("Logistic Regression:", accuracy_score(y_test, lr.predict(X_test)))
print("SVM:", accuracy_score(y_test, svm.predict(X_test)))
print("MLP:", accuracy_score(y_test, mlp.predict(X_test)))
print("Ensemble:", accuracy_score(y_test, y_ensemble))

100%|██████████| 170M/170M [00:05<00:00, 28.8MB/s] 


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Logistic Regression: 0.792
SVM: 0.8147
MLP: 0.8054
Ensemble: 0.82
